In [ ]:
# Nonsym CoV-beam best_rep → length-greedy @ 10k — CONFIG
# Edit ONLY this cell. Restart → Run All continues (resume on jsonl).
# Starts = Aut-min best_rep from covbeam_nonsym_b1kfull_subset60.jsonl.
# Default IDs = the 11 not proven at greedy@1k (49/60 already done).

REPO_URL   = "https://github.com/Avi161/ACSolverX.git"
REPO_DIR   = "ACSolverX"
BRANCH     = "cursor/heur-u124-s20mk2-a42e"
CLONE      = True
UPDATE_REPO = True

MOUNT_DRIVE = True
DRIVE_DIR   = "/content/drive/MyDrive/acsolverx/covbeam_nonsym_b10k"

NODE_BUDGET = 10_000
# None = all 60; default = the 11 open after greedy@1k
IDS = "596,605,610,622,623,624,625,636,637,638,639"
OUT_STEM = "covbeam_nonsym_g10000_from_climb_subset60"
STAGE_DIR = "/content/covbeam_stage"


In [ ]:
# ==================== SETUP (clone / pull / mount / purge) =================
import os, sys, subprocess

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    os.makedirs(DRIVE_DIR, exist_ok=True)

BASE = "/content" if IN_COLAB else os.getcwd()
os.chdir(BASE)
if CLONE:
    if not os.path.isdir(REPO_DIR):
        sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch origin {BRANCH} && git reset --hard FETCH_HEAD")

REPO = os.path.join(BASE, REPO_DIR)
os.chdir(REPO)
sys.path.insert(0, REPO)
for k in list(sys.modules):
    if k == "experiments" or k.startswith("experiments."):
        del sys.modules[k]

os.makedirs(STAGE_DIR, exist_ok=True)
src = os.path.join(DRIVE_DIR, f"{OUT_STEM}.jsonl")
dst_dir = os.path.join(REPO, "results", "comparison")
os.makedirs(dst_dir, exist_ok=True)
dst = os.path.join(dst_dir, f"{OUT_STEM}.jsonl")
if os.path.exists(src) and not os.path.exists(dst):
    import shutil
    shutil.copy2(src, dst)
    print("seeded", dst, "from Drive")
print("BRANCH:", BRANCH, "BUDGET:", NODE_BUDGET, "IDS:", IDS)


In [ ]:
# ==================== RUN ==================================================
import os, shutil, time
from experiments.heuristic_search.runners.run_covbeam_greedy_from_climb import run as run_greedy

ids = [s.strip() for s in IDS.split(",") if s.strip()] if IDS else None
t0 = time.time()
out_jsonl, out_csv = run_greedy(
    NODE_BUDGET, ids=ids, out_stem=OUT_STEM, allow_over_local_cap=True)
print(f"wall {time.time()-t0:.1f}s")

if IN_COLAB and MOUNT_DRIVE:
    for p in (out_jsonl, out_csv):
        if p and os.path.exists(p):
            shutil.copy2(p, os.path.join(DRIVE_DIR, os.path.basename(p)))
            print("mirrored", os.path.basename(p), "→", DRIVE_DIR)
